# 17 — Command-Line Interfaces and Project Packaging

Turn notebook logic into reusable Python modules and a command-line program. argparse parses arguments, while pyproject.toml describes a distributable project. Keep executable entry points thin and core behavior testable.

*Intermediate · 75–120 min*

## You will learn

- Parse CLI arguments with argparse
- Design functions that can be called without command-line access
- Describe a src-layout package with pyproject.toml
- Distinguish dependencies, entry points, and modules

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

## Examples

### 1. argparse basics

argparse declares flags, positional arguments, choices, defaults, and help text. Passing an explicit list of arguments in a notebook avoids accidentally parsing Jupyter's kernel options. A real script calls parse_args() without a list.

In [ ]:
import argparse

parser = argparse.ArgumentParser(description="Summarize a score")
parser.add_argument("score", type=float)
parser.add_argument("--pass-mark", type=float, default=70)
args = parser.parse_args(["86.5", "--pass-mark", "75"])
print(args.score)
print(args.pass_mark)

**Think about it:** What does argparse do when a user passes text where a float is required?

### 2. Separate parsing, domain logic, and output

A command-line interface should convert raw arguments and call regular functions. Keep the calculation testable without launching a subprocess. Use explicit return values instead of terminating the process from deep inside business logic.

In [ ]:
def passing(score, threshold):
    if not 0 <= score <= 100 or not 0 <= threshold <= 100:
        raise ValueError("scores and thresholds must be between 0 and 100")
    return score >= threshold

def format_result(score, threshold):
    return "PASS" if passing(score, threshold) else "REVIEW"

assert format_result(82, 70) == "PASS"
print(format_result(args.score, args.pass_mark))

**Think about it:** Why is passing() easier to unit-test than a function that always reads sys.argv?

### 3. Module layout and pyproject.toml

A typical package uses a src directory, a named Python package, tests, a README, and a pyproject.toml. The pyproject file describes metadata, dependencies, build backend, and optional console-script entry points. This is a demonstration string, not a file write.

In [ ]:
project_tree = """
demo-project/
  pyproject.toml
  README.md
  src/demo_project/__init__.py
  src/demo_project/cli.py
  src/demo_project/core.py
  tests/test_core.py
"""
pyproject_example = """
[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "demo-project"
version = "0.1.0"
requires-python = ">=3.10"

[project.scripts]
demo-report = "demo_project.cli:main"
"""
print(project_tree)
print(pyproject_example)

**Think about it:** Why is an entry point mapped to a function rather than to a notebook cell?

### 4. Entry points and return codes

An executable CLI normally defines main(argv=None), catches expected input/domain errors, prints a useful message, and returns an exit status. The if __name__ == '__main__' guard belongs in script files, not in a tutorial cell that should remain passive.

In [ ]:
def main(argv=None):
    parser = argparse.ArgumentParser(description="Test a grade")
    parser.add_argument("grade", type=float)
    parsed = parser.parse_args(argv)
    try:
        message = format_result(parsed.grade, 70)
    except ValueError as exc:
        print("Error:", exc)
        return 2
    print(message)
    return 0

print("Exit code:", main(["94"]))
# In a .py file, call raise SystemExit(main()) under the __main__ guard.

**Think about it:** What does an exit status other than zero communicate to a shell script?

## Try it

Short fill-in exercises on this lesson: replace each `___`, run the cell, and `check` tells you whether you're right. The answer itself stays hidden. More practice for this topic is in the [workbook](../exercises/04_sql_http_cli_performance_exercises.ipynb).

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/17', {1: 'a3cd5baed6c048bc', 2: 'b99604c3fed984fc', 3: 'ebec440e01139af5', 4: '5f46f3ee84fabf6a'})

**Exercise 1.** argparse arguments for a tax calculator.

In [ ]:
import argparse
parser = argparse.ArgumentParser(prog="tax")
parser.add_argument("amount", type=___)
parser.add_argument("--rate", type=float, default=___)
args = parser.parse_args(["100"])
check(1, (args.amount, args.rate))

**Exercise 2.** A pure core function and a thin `main(argv)` wrapper.

In [ ]:
def add_tax(amount, rate):
    return round(amount * (1 + rate), 2)

def main(argv):
    amount, rate = float(argv[0]), float(argv[1])
    return f"{___:.2f}"

check(2, [add_tax(10, 0.2), main(["10", "0.05"])])

**Exercise 3.** Which `pyproject.toml` table declares a console command?

In [ ]:
import tomllib
text = """
[project]
name = "taxcalc"
version = "1.0.0"

[project.scripts]
taxcalc = "taxcalc.cli:main"
"""
data = tomllib.loads(text)
check(3, data["project"][___])

**Exercise 4.** Test `main` by passing an explicit argument list.

In [ ]:
import argparse

def main(argv):
    parser = argparse.ArgumentParser()
    parser.add_argument("name")
    parser.add_argument("--shout", action=___)
    args = parser.parse_args(argv)
    greeting = f"hello {args.name}"
    return greeting.upper() if args.shout else greeting

check(4, [main(["ada"]), main(___)])

## Common mistakes

- Do not call parse_args() with Jupyter's sys.argv by accident.
- Avoid raising SystemExit from ordinary lesson code.
- Packaging configuration is project metadata, not an automatic replacement for testing.

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** argparse `action='store_true'` makes a flag…

   - **a)** Require a value
   - **b)** Repeatable
   - **c)** True when present, False otherwise

**2.** Why keep CLI parsing separate from the core logic?

   - **a)** argparse is slow
   - **b)** It's required by pyproject
   - **c)** The logic can be tested and reused without the command line

**3.** In a src layout, the package lives in…

   - **a)** The repo root
   - **b)** tests/
   - **c)** src/<package>/

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/17', {1: 'e91a6438f3cb', 2: 'aa8076a1fe5f', 3: '5b95a1f09499'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [18 — Algorithmic Complexity, Profiling, and Optimization](18_complexity_profiling_and_optimization.ipynb)